# Setup

In [1]:
from stat_test_tools import *

descriptions = {
    "max_fit": "Fitness (max fitness)",
    "div": "Behavioral diversity"
}

container_naming = {
    "add_novelty": "inc. fitness",
    "sub_novelty": "inc. novelty",
    "novelty": "pure novelty",
    "novelty_archiving": "novetly archive",
    "novelty_limit": "limit archive",
    "fit_archiving": "fit. archive",
    "elite_archiving": "elite archive",
    "fitness":"fitness"
}

env_naming = {
    "lunarlander": "Lunar Lander",
    "cartpole": "Cartpole"
}

alg_naming = {
    "lambda": "ES",
    "diff": "DE"
}

hyperparameter_naming = {
    "m": "$\lambda$",
    "l": "$\mu$",
    "mutation_sigma": "$\sigma$",
    "archive_batch": "arch. batch",
    "archiving_period": "arch. T",
    "cross_method": "method",
    "fitness_weight":"$W_0$",
    "decay":"$r_D$",
    "cross_uni": "$p_U$"
}

datasets = {"cartpole":"server_try0", "lunarlander":"server_try2"}
def fmt(x):
    if pd.isna(x):
        return "-"
    if isinstance(x, (int, float)):
        if x == 0:
            return "0"
        elif abs(x) < 1e-3:
            mantissa, exp = f"{x:.2e}".split("e")
            return rf"${mantissa}\times 10^{{{int(exp)}}}$"
        else:
            return f"{x:.2f}"
    if x in container_naming:
        return container_naming[x]
    return str(x)

2026-07-03 14:40:46.580892: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-03 14:40:47.584712: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/schkliba/.prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Grid protocols

In [42]:
ens = ["lunarlander", "cartpoles"]
en = "lunarlander"
alg = "diff"
df = get_protocol_table(en, alg, datasets[en])
#df.to_csv(f"./Data/grid_protocol/{en}_{alg}.csv")
dk = df["origin"].copy()
dk


Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
[{'lambda': 50, 'mutation_rate': 0.8, 'cross_rate': 0.8, 'archiving_period': 2, 'archive_batch': 1, 'limit': -180.0}]
{'l': 60, 'mr': 0.9, 'cr': 0.8, 'archiving_period': 2, 'archive_batch': 1, 'limit': -180.0}


,archive_batch,archiving_period,cr,decay,fitness_weight,l,limit,mr
fitness,NaN,NaN,0.3,NaN,NaN,40.0,NaN,0.06
novelty,NaN,NaN,0.4,NaN,NaN,40.0,NaN,0.15
add_novelty,NaN,NaN,0.4,4.0,0.5,40.0,NaN,0.13
sub_novelty,NaN,NaN,0.6,1.0,0.4,70.0,NaN,0.04
fit_archiving,2.0,2.0,0.6,NaN,NaN,40.0,NaN,0.30
elite_archiving,4.0,5.0,0.5,NaN,NaN,40.0,NaN,0.30
novelty_archiving,5.0,4.0,0.9,NaN,NaN,60.0,NaN,0.00
novelty_limit,1.0,2.0,0.8,NaN,NaN,50.0,-180.0,0.80


In [43]:
# from tabulate import tabulate
# df2 = pd.read_csv(f"./Data/grid_protocol/{en}_{alg}.csv", header=[0,1], index_col=[0])
# if alg =="lambda":
#     df2=df2.drop([("final","cross_method"), ("origin", "cross_method")], axis=1)
# delta =  df2["final"] - df2["origin"]
# delta = delta.loc[~(delta.fillna(0) == 0).all(axis=1)]
# #delta = delta.loc[~(delta.fillna(0) == 0).all()]
# mask = ~delta.fillna(0).eq(0).all(axis=0)
# delta = delta.loc[:, mask]

# delta.to_csv(f"./Data/grid_protocol/delta_{en}_{alg}.csv")
def table_to_latex(dk):
    dk = dk.replace(["NaN", "nan", None], np.nan)
    new_columns = [hyperparameter_naming[col] if col in hyperparameter_naming else col for col in dk.columns]
    dk.columns = new_columns
    index = [container_naming[col] if col in container_naming else col for col in dk.index]
    dk.index = index
    dk=dk.fillna("-")
    if "method" in dk.columns:
        dk["method"] = dk["method"].apply(lambda x: "A" if x =="mean" else "U")
    latex = dk.to_latex(
        index=True,
        escape=False,
        formatters= [fmt] * len(dk.columns),
        column_format="l" + "c" * len(dk.columns)
    )
    return latex

latex = table_to_latex(dk)
print(latex)

\begin{tabular}{lcccccccc}
\toprule
{} & arch. batch & arch. T &   cr & $r_D$ & $W_0$ & $\mu$ &  limit &   mr \\
\midrule
fitness         &           - &       - & 0.30 &     - &     - & 40.00 &      - & 0.06 \\
pure novelty    &           - &       - & 0.40 &     - &     - & 40.00 &      - & 0.15 \\
inc. fitness    &           - &       - & 0.40 &   4.0 &   0.5 & 40.00 &      - & 0.13 \\
inc. novelty    &           - &       - & 0.60 &   1.0 &   0.4 & 70.00 &      - & 0.04 \\
fit. archive    &         2.0 &     2.0 & 0.60 &     - &     - & 40.00 &      - & 0.30 \\
elite archive   &         4.0 &     5.0 & 0.50 &     - &     - & 40.00 &      - & 0.30 \\
novetly archive &         5.0 &     4.0 & 0.90 &     - &     - & 60.00 &      - &    0 \\
limit archive   &         1.0 &     2.0 & 0.80 &     - &     - & 50.00 & -180.0 & 0.80 \\
\bottomrule
\end{tabular}



/tmp/ipykernel_380895/3087266357.py:21: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = dk.to_latex(


# Gens graph

In [24]:
import constants as cst
en = "lunarlander"
alg = "diff"
names = {"lunarlander":"server_try2", "cartpole":"server_try0"}

def create_figures(names, attr):
    path = "./Docs/vzor-dp/img/"
    for alg in ["lambda", "diff"]:
        for en in cst.ENIVROMENTS:
            plt.figure()
            ax = plot_generations(en=en, alg=alg, name=names[en], attr=attr)
            handles, labels = ax.get_legend_handles_labels()
            new_labels = [container_naming[l] for l in labels]
            ax.legend_.remove()
            ax.set_xlabel("Generation (ng)")
            ax.set_ylabel(descriptions[attr])
            plt.legend(handles, new_labels, loc="center left", bbox_to_anchor=(1, 0.5))
            plt.title(f"{env_naming[en]}-{alg_naming[alg]}")
            plt.savefig(path + f"generations_{en}_{alg}_{attr}.pdf", bbox_inches="tight")

create_figures(names, "max_fit")

NameError: name 'plt' is not defined

In [22]:
print(df.columns.tolist())

['seed', 'ng', 'population', 'max_fit', 'avg_fit', 'median_fit', 'min_fit', 'std_fit', 'div', 'cont']


In [3]:
df = gen_extract_values(load_gen_direct_data("cartpole", "fitness", "lambda", "server_try0"))
df.groupby("ng").median()

Found 1 files!


/tmp/ipykernel_22693/2083778437.py:2: FutureWarning: The default value of numeric_only in DataFrameGroupBy.median is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df.groupby("ng").median()


,max_fit,avg_fit,median_fit,min_fit,std_fit,div
ng,,,,,,
5,380.0,248.844444,182.166667,38.333333,137.925233,0.547401
10,500.0,320.355556,380.000000,23.666667,150.994717,0.391541
15,500.0,238.411111,181.500000,14.166667,152.964761,0.386958
20,500.0,215.944444,245.666667,9.833333,136.345529,0.549736
25,500.0,223.366667,224.000000,25.500000,144.079880,0.527765


# Statistical tests

In [ ]:
cont = "fitness"
alg = "lambda"



## Algorithms

In [3]:
total = wilcoxon_test_total_algorithms(datasets, "fitnesses")
total

Found 1 files!
Found 2 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 2 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 2 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 2 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!


(60862.0, 0.22415735550520588)

In [4]:
ll_wilcoxon = wilcoxon_test_per_environment_algorithms("lunarlander",datasets, "fitnesses")
carpole_wilcoxon = wilcoxon_test_per_environment_algorithms("cartpole",datasets, "fitnesses")

-68.80925021233728
29.124999999999996


In [5]:
ll_wilcoxon

(7434.0, 6.768479997832337e-11, -68.80925021233728)

In [6]:
carpole_wilcoxon

(13428.0, 0.0002129582820475215, 29.124999999999996)

## Containers

### Friedman

In [3]:
f_c = friedman_test_per_environment_containers("cartpole", datasets, "fitnesses")
f_l =friedman_test_per_environment_containers("lunarlander", datasets, "fitnesses")
print(f_c)
print(f_l)

FriedmanchisquareResult(statistic=326.65791750793215, pvalue=1.2158324856403088e-66)
FriedmanchisquareResult(statistic=210.29033538400478, pvalue=7.573222685209835e-42)


In [6]:
f_c = friedman_test_per_environment_containers("cartpole", datasets, "behaviors")
f_l =friedman_test_per_environment_containers("lunarlander", datasets, "behaviors")
print(f_c)
print(f_l)

FriedmanchisquareResult(statistic=163.91859695565856, pvalue=4.7999219391876454e-32)
FriedmanchisquareResult(statistic=192.23458038422646, pvalue=5.052927393799729e-38)


In [8]:
def wilcoxon_to_latex(dk, contestant_names):
    dk = dk.copy()
    assert (dk["holm_reject"] == dk["hoch_reject"]).all()
    dk["Conclusion"] = dk["holm_reject"].apply(lambda x: "rejected" if x else "sustained")
    if "direction" in dk:
        dk["direction"] = dk[["holm_reject", "direction"]].apply(lambda x: x["direction"] if x["holm_reject"] else "-", axis=1)
    dk["contestant1"] = dk["contestant1"].apply(lambda x: contestant_names[x])
    dk["contestant2"] = dk["contestant2"].apply(lambda x: contestant_names[x])

    COLUMN_NAMES = {
        "direction":"HL",
        "hoch_p": "$p_{Hoch}$",
        "holm_p": "$p_{Holm}$",
        "wilcoxon_T": "$W_T$",
        "contestant1": "A",
        "contestant2": "B"
    }

    new_columns = [COLUMN_NAMES[col] if col in COLUMN_NAMES else col for col in dk.columns]
    dk.columns = new_columns

    
    dk = dk.drop(columns=["holm_reject", "hoch_reject"])
    latex = dk.to_latex(
        index=False,
        escape=False,
        formatters= [fmt] * len(dk.columns),
        column_format="l" + "c" * len(dk.columns)
    )
    return latex

In [16]:
ll_wilcoxon = apply_corrections(wilcoxon_test_per_environment_containers("lunarlander",datasets, "fitnesses"))
carpole_wilcoxon = apply_corrections(wilcoxon_test_per_environment_containers("cartpole",datasets, "fitnesses"))
l = wilcoxon_to_latex(ll_wilcoxon,container_naming)
c = wilcoxon_to_latex(carpole_wilcoxon, container_naming)
print(c)

\begin{tabular}{lcccccccc}
\toprule
              A &               B &    $W_T$ &                     p &          HL &            $p_{Holm}$ &            $p_{Hoch}$ & Conclusion \\
\midrule
        fitness &    pure novelty &   50.000 & $3.85\times 10^{-13}$ &  179.666667 & $6.54\times 10^{-12}$ & $6.54\times 10^{-12}$ &   rejected \\
        fitness &    inc. fitness &  198.500 &  $2.20\times 10^{-7}$ &   79.916667 &  $2.86\times 10^{-6}$ &  $2.86\times 10^{-6}$ &   rejected \\
        fitness &    inc. novelty &   66.000 & $4.01\times 10^{-14}$ &  195.416667 & $8.41\times 10^{-13}$ & $8.41\times 10^{-13}$ &   rejected \\
        fitness &    fit. archive &  324.000 &                 0.111 &           - &                 0.444 &                 0.444 &  sustained \\
        fitness &   elite archive &  138.500 &                 0.088 &           - &                 0.438 &                 0.438 &  sustained \\
        fitness & novetly archive &   11.000 & $1.73\times 10^{-15}$ &   

/tmp/ipykernel_377701/4228006561.py:32: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = dk.to_latex(
/tmp/ipykernel_377701/4228006561.py:32: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = dk.to_latex(


In [10]:
ll_wilcoxon = apply_corrections(wilcoxon_test_per_environment_containers("lunarlander",datasets, "behaviors"))
carpole_wilcoxon = apply_corrections(wilcoxon_test_per_environment_containers("cartpole",datasets, "behaviors"))
l = wilcoxon_to_latex(ll_wilcoxon,container_naming)
c = wilcoxon_to_latex(carpole_wilcoxon, container_naming)
print(l)

\begin{tabular}{lcccccccc}
\toprule
              A &               B &  $W_T$ &                     p &        HL &            $p_{Holm}$ &            $p_{Hoch}$ & Conclusion \\
\midrule
        fitness &    pure novelty &  49.00 & $1.83\times 10^{-10}$ & -0.102046 &  $4.20\times 10^{-9}$ &  $4.20\times 10^{-9}$ &   rejected \\
        fitness &    inc. fitness & 119.00 &  $1.18\times 10^{-8}$ & -0.069382 &  $2.60\times 10^{-7}$ &  $2.60\times 10^{-7}$ &   rejected \\
        fitness &    inc. novelty & 122.00 &  $2.18\times 10^{-8}$ & -0.075654 &  $4.57\times 10^{-7}$ &  $4.57\times 10^{-7}$ &   rejected \\
        fitness &    fit. archive & 501.00 &                  0.04 &         - &                  0.26 &                  0.26 &  sustained \\
        fitness &   elite archive &  59.00 &  $1.44\times 10^{-7}$ &  0.066169 &  $2.44\times 10^{-6}$ &  $2.44\times 10^{-6}$ &   rejected \\
        fitness & novetly archive & 273.00 &  $2.29\times 10^{-6}$ & -0.119972 &  $3.43\times 10^

/tmp/ipykernel_382690/2025726831.py:24: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = dk.to_latex(
/tmp/ipykernel_382690/2025726831.py:24: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = dk.to_latex(


In [48]:
df = wilcoxon_test_total_containers(datasets, "fitnesses")
dk = apply_corrections(df)


latex = wilcoxon_to_latex(dk, container_naming)
print(latex)

\begin{tabular}{lccccccc}
\toprule
              A &               B &   $W_T$ &                     p &            $p_{Holm}$ &            $p_{Hoch}$ & Conclusion \\
\midrule
        fitness &    pure novelty &  412.00 & $4.47\times 10^{-20}$ & $7.16\times 10^{-19}$ & $7.16\times 10^{-19}$ &   rejected \\
        fitness &    inc. fitness & 1970.50 &  $3.54\times 10^{-5}$ &  $3.19\times 10^{-4}$ &  $3.19\times 10^{-4}$ &   rejected \\
        fitness &    inc. novelty &  511.00 & $1.44\times 10^{-20}$ & $2.45\times 10^{-19}$ & $2.45\times 10^{-19}$ &   rejected \\
        fitness &    fit. archive & 2598.00 &                  0.92 &                  1.00 &                  0.99 &  sustained \\
        fitness &   elite archive & 1465.50 &                  0.03 &                  0.11 &                  0.11 &  sustained \\
        fitness & novetly archive &  137.00 & $2.54\times 10^{-24}$ & $6.35\times 10^{-23}$ & $6.35\times 10^{-23}$ &   rejected \\
        fitness &   limit archiv

/tmp/ipykernel_380895/2025726831.py:24: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = dk.to_latex(


In [ ]:

friedman_test_total_containers( datasets, "fitnesses")

Found 1 files!
Found 2 files!
Found 1 files!
Found 2 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 2 files!
Found 1 files!
Found 2 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!


FriedmanchisquareResult(statistic=517.6305019627496, pvalue=1.297364879063882e-107)

In [ ]:
friedman_test_total( "lambda", {"cartpole":"server_try0", "lunarlander":"server_try2"}, "fitnesses")


NameError: name 'friedman_test_total' is not defined

# Hyperparameters Optuna

## Lambda

### Cartpole

cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
l = trial.suggest_categorical("lambda",[10,20,30])
m = trial.suggest_categorical("mu",[10,20,30])
mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
sigma = trial.suggest_float("sigma", 0.5, 2, step=0.5)
archiving = trial.suggest_int("archiving_period", 2, 5)
archive_batch = trial.suggest_int("archive_batch", 1, 5)
limit = trial.suggest_float("limit", 200, 300, step=10)
fit_w = trial.suggest_float("start_fit_w", 0.2, 0.7, step=0.1)
decay = trial.suggest_float("decay", 0.5, 5, step=0.5)
cross_uni = None
if cross_method == "uniform":
    cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)




In [47]:
import pandas as pd
en = "cartpole"
alg = "lambda"
import pandas as pd

cols = pd.MultiIndex.from_product(
    [[en], ["Min", "Max", "Step"]]
)

df = pd.DataFrame([
    ["uniform", "mean", "-"],
    [10, 30, 10],
    [10, 30, 10],
    [0.00, 0.10, 0.01],
    [0.3, 1.0, 0.1],
    [0.5, 2.0, 0.5],
    [2, 5, 1],
    [1, 5, 1],
    [200, 300, 10],
    [0.2, 0.7, 0.1],
    [0.5, 5.0, 0.5],
    [0.1, 0.9, 0.1],
],
index=[
    "cross_method",
    "lambda",
    "mu",
    "mutation_rate",
    "cross_rate",
    "sigma",
    "archiving_period",
    "archive_batch",
    "limit",
    "start_fit_w",
    "decay",
    "cross*"
],
columns=cols)
df_c = df#.reset_index().rename(columns={"index": "Parameter"})

# df.to_csv(f"./Data/bayes_protocol/params_{alg}.csv")
# latex = df.to_latex(
#     index=False,
#     escape=True,
#     float_format=lambda x: f"{x:.3f}",
#     column_format="l" + "c" * len(df.columns)
# )
# print(latex)

### Lunar Lander

cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
m = trial.suggest_categorical("mu",[40, 50, 60, 70])
mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
decay = trial.suggest_float("decay", 0.5, 5, step=0.5)
archiving = trial.suggest_int("archiving_period", 2, 5)
archive_batch = trial.suggest_int("archive_batch", 1, 5)
limit = trial.suggest_float("limit", -200, -50, step=10)

cross_uni = None
if cross_method == "uniform":
    cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)

In [50]:
en="lunarlander"
columns = pd.MultiIndex.from_product([[en], ["Min", "Max", "Step"]])

df = pd.DataFrame(
    [
        ["uniform", "mean", "-"],
        [40, 70, 10],
        [40, 70, 10],
        [0.00, 0.50, 0.01],
        [0.3, 1.0, 0.1],
        [0.5, 3.0, 0.5],
        [0.2, 1.0, 0.1],
        [0.5, 5.0, 0.5],
        [2, 5, 1],
        [1, 5, 1],
        [-200, -50, 10],
        [0.1, 0.9, 0.1],
    ],
    index=[
        "cross_method",
        "lambda",
        "mu",
        "mutation_rate",
        "cross_rate",
        "sigma",
        "start_fit_w",
        "decay",
        "archiving_period",
        "archive_batch",
        "limit",
        "cross*",
    ],
    columns=columns,
    )
df_l = df
#df_l = df.reset_index().rename(columns={"index": "Parameter"})

df_total = df_c.join(df_l)
df = df_total.reset_index().rename(columns={"index": "Parameter"})

df.to_csv(f"./Data/bayes_protocol/params_{alg}.csv")
latex = df.to_latex(
    index=False,
    escape=True,
    float_format=lambda x: f"{x:.2f}",
    column_format="l" + "c" * len(df.columns)
)
print(latex)

\begin{tabular}{lccccccc}
\toprule
       Parameter & \multicolumn{3}{l}{cartpole} & \multicolumn{3}{l}{lunarlander} \\
                 &      Min &  Max & Step &         Min &  Max & Step \\
\midrule
    cross\_method &  uniform & mean &    - &     uniform & mean &    - \\
          lambda &       10 &   30 &   10 &          40 &   70 &   10 \\
              mu &       10 &   30 &   10 &          40 &   70 &   10 \\
   mutation\_rate &     0.00 & 0.10 & 0.01 &        0.00 & 0.50 & 0.01 \\
      cross\_rate &     0.30 & 1.00 & 0.10 &        0.30 & 1.00 & 0.10 \\
           sigma &     0.50 & 2.00 & 0.50 &        0.50 & 3.00 & 0.50 \\
archiving\_period &        2 &    5 &    1 &           2 &    5 &    1 \\
   archive\_batch &        1 &    5 &    1 &           1 &    5 &    1 \\
           limit &      200 &  300 &   10 &        -200 &  -50 &   10 \\
     start\_fit\_w &     0.20 & 0.70 & 0.10 &        0.20 & 1.00 & 0.10 \\
           decay &     0.50 & 5.00 & 0.50 &        0.50 & 5.0

/tmp/ipykernel_339521/1110227470.py:42: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = df.to_latex(


## Differential

### Carpole

l = trial.suggest_categorical("pop",[5,10,15,20])
mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
archiving = trial.suggest_int("archiving_period", 2, 5)
archive_batch = trial.suggest_int("archive_batch", 1, 5)
limit = trial.suggest_float("limit", 200, 300, step=10)
fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
decay = trial.suggest_float("decay", 0.5, 5, step=0.5)


In [51]:
en = "cartpole"
columns = pd.MultiIndex.from_product([[en], ["Min", "Max", "Step"]])

df = pd.DataFrame(
    [
        [5, 20, 5],          # pop
        [0.0, 1.0, 0.1],     # mutation_rate
        [0.3, 1.0, 0.1],     # cross_rate
        [2, 5, 1],           # archiving_period
        [1, 5, 1],           # archive_batch
        [200, 300, 10],      # limit
        [0.2, 1.0, 0.1],     # start_fit_w
        [0.5, 5.0, 0.5],     # decay
    ],
    index=[
        "pop",
        "mutation_rate",
        "cross_rate",
        "archiving_period",
        "archive_batch",
        "limit",
        "start_fit_w",
        "decay",
    ],
    columns=columns
)

df_c = df

### Lunar lander

l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
archiving = trial.suggest_int("archiving_period", 2, 5)
archive_batch = trial.suggest_int("archive_batch", 1, 5)
limit = trial.suggest_float("limit", -200, -50, step=10)
fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

In [53]:
en = "lunarlander"
alg = "diff"
columns = pd.MultiIndex.from_product([[en], ["Min", "Max", "Step"]])

df = pd.DataFrame(
    [
        [40, 70, 10],        # pop
        [0.0, 1.0, 0.1],     # mutation_rate
        [0.3, 1.0, 0.1],     # cross_rate
        [2, 5, 1],           # archiving_period
        [1, 5, 1],           # archive_batch
        [-200, -50, 10],     # limit
        [0.2, 1.0, 0.1],     # start_fit_w
        [0.5, 5.0, 0.5],     # decay
    ],
    index=[
        "pop",
        "mutation_rate",
        "cross_rate",
        "archiving_period",
        "archive_batch",
        "limit",
        "start_fit_w",
        "decay",
    ],
    columns=columns
)
df_l = df
df_total = df_c.join(df_l)
df = df_total.reset_index().rename(columns={"index": "Parameter"})

df.to_csv(f"./Data/bayes_protocol/params_{alg}.csv")
latex = df.to_latex(
    index=False,
    escape=True,
    float_format=lambda x: f"{x:.2f}",
    column_format="l" + "c" * len(df.columns)
)
print(latex)

\begin{tabular}{lccccccc}
\toprule
       Parameter & \multicolumn{3}{l}{cartpole} & \multicolumn{3}{l}{lunarlander} \\
                 &      Min &    Max &  Step &         Min &    Max &  Step \\
\midrule
             pop &     5.00 &  20.00 &  5.00 &       40.00 &  70.00 & 10.00 \\
   mutation\_rate &     0.00 &   1.00 &  0.10 &        0.00 &   1.00 &  0.10 \\
      cross\_rate &     0.30 &   1.00 &  0.10 &        0.30 &   1.00 &  0.10 \\
archiving\_period &     2.00 &   5.00 &  1.00 &        2.00 &   5.00 &  1.00 \\
   archive\_batch &     1.00 &   5.00 &  1.00 &        1.00 &   5.00 &  1.00 \\
           limit &   200.00 & 300.00 & 10.00 &     -200.00 & -50.00 & 10.00 \\
     start\_fit\_w &     0.20 &   1.00 &  0.10 &        0.20 &   1.00 &  0.10 \\
           decay &     0.50 &   5.00 &  0.50 &        0.50 &   5.00 &  0.50 \\
\bottomrule
\end{tabular}



/tmp/ipykernel_339521/3449472945.py:33: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = df.to_latex(


# Generations Results

In [11]:
carpole_dict = {
    "sub_novelty":{
        "lambda": 20,
        "diff":40
    },
    "add_novelty":{
        "diff": 20,
        "lambda": 15
    },
    "novelty":{
        "lambda":25,
        "diff": 40
    },
    "fitness":{
        "diff": 10,
        "lambda": 15
    },
    "fit_archiving":{
        "diff": 25,
        "lambda": 35
    },
    "elite_archiving":{
        "lambda": 20,
        "diff": 10
    },
    "novelty_limit":{
        "lambda": 20,
        "diff": 20
    },
    "novelty_archiving":{
        "diff": 20,
        "lambda": 20
    }
}
lunarlander = {
    "fitness":{
        "diff": 35,
        "lambda": 25
    },
    "novelty":{
        "lambda": 10,
        "diff": 15
    },
    "fit_archiving":{
        "diff": 80,
        "lambda": 15,
    },
    "add_novelty":{
        "lambda": 15,
        "diff":15
    },
    "sub_novelty":{
        "lambda": 25,
        "diff": 25
    },
    "novelty_archiving":{
        "diff": 25,
        "lambda": 60
    },
    "elite_archiving":{
        "lambda":15,
        "diff": 60,
    },
    "novelty_limit":{
        "lambda":25,
        "diff": 25
    }
}

In [ ]:
luna = pd.DataFrame(lunarlander).T
cartpole = pd.DataFrame(carpole_dict).T
luna_new_columns = [alg_naming[col] if col in hyperparameter_naming else col for col in luna.columns]
cart_new_columns = [alg_naming[col] if col in hyperparameter_naming else col for col in luna.columns]

luna.columns = pd.MultiIndex.from_product([["Lunar Lander"], luna_new_columns])
cartpole.columns = pd.MultiIndex.from_product([["Cartpole"], cart_new_columns])
gens_table = pd.concat([luna, cartpole], axis=1)

Lunar Lander        Cartpole     
                          diff lambda   lambda diff
fitness                     35     25       15   10
novelty                     15     10       25   40
fit_archiving               80     15       35   25
add_novelty                 15     15       15   20
sub_novelty                 25     25       20   40
novelty_archiving           25     60       20   20
elite_archiving             60     15       20   10
novelty_limit               25     25       20   20

In [ ]:
def table_to_latex(dk):
    dk = dk.replace(["NaN", "nan", None], np.nan)
    new_columns = [hyperparameter_naming[col] if col in hyperparameter_naming else col for col in dk.columns]
    dk.columns = new_columns
    index = [container_naming[col] if col in container_naming else col for col in dk.index]
    dk.index = index
    dk=dk.fillna("-")
    if "method" in dk.columns:
        dk["method"] = dk["method"].apply(lambda x: "A" if x =="mean" else "U")
    latex = dk.to_latex(
        index=True,
        escape=False,
        formatters= [fmt] * len(dk.columns),
        column_format="l" + "c" * len(dk.columns)
    )
    return latex